# JijZept Solverで最適化問題を解く

`jijmodeling` の使い方を理解するために、このページではナップサック問題を解いてみましょう。ただし、`jijmodeling` は数理モデルを記述するためのツールであるため、単独では最適化問題を解くことはできません。なので、Jij 製の数理最適化ソルバー [JijZept Solver](https://www.jijzept.com/ja/products/solver/) の無償 WebAPI 版と組み合わせて解くこととします。

このノートブックは、JijModeling Tutorials の [OpenJijで最適化問題を解く](https://jij-inc-jijmodeling-tutorials-ja.readthedocs-hosted.com/ja/latest/quickstart/openjij.html) を JijZept Solver 向けに書き換えたものです。

`jijmodeling` と JijZept Solver を組み合わせて使うには、JijZept Solver のクライアントパッケージ `jijzept-solver` ([PyPI](https://pypi.org/project/jijzept-solver/)) をインストールする必要があります。以下のコマンドでインストールしてください（Binder 上では既にインストールされています）。

```bash
pip install jijmodeling jijzept-solver
```

> **アクセス情報について**
>
> JijZept Solver（無償 WebAPI 版）を使うには、事前に [利用申請フォーム](https://docs.google.com/forms/d/e/1FAIpQLScLTRxXGaN7egRkoYcq2ZvFoFXRyYInsmPXlyxk9pF11E9--g/viewform) から申し込み、メールで届く **API サーバーのホスト名** と **アクセストークン** を入手してください。これらは他人に共有したり、書き込んだノートブックを公開したりしないでください。

## JijZept Solver 環境設定

In [ ]:
import os

os.environ["JIJZEPT_SOLVER_SERVER_HOST"] = "APIサーバーのホスト名"
os.environ["JIJZEPT_SOLVER_ACCESS_TOKEN"] = "アクセストークン"

## 問題設定

ナップサック問題は以下のように数理モデルとして定式化することができます：

$$
\begin{align*}
\mathrm{maximize} \quad & \sum_{i=0}^{N-1} v_i x_i \\
\mathrm{s.t.} \quad & \sum_{i=0}^{N-1} w_i x_i \leq W, \\
& x_{i} \in \{ 0, 1\}
\end{align*}
$$

> **ヒント**: ナップサック問題の定式化について詳しく知りたい場合は [JijZeptチュートリアル](https://jij-inc.github.io/JijZept-Tutorials/ja/src/02_knapsack.html) を参照してください。

この数理モデルにあるそれぞれのパラメーターの意味は以下の通りです：

| パラメーター | 説明 |
| --- | --- |
| $N$ | アイテムの総数 |
| $v_{i}$ | アイテム $i$ の価値 |
| $w_{i}$ | アイテム $i$ の重さ |
| $W$ | ナップサックの耐荷重 |

今回の説明では、上記の数理モデルのパラメーター $v_{i}, w_{i}, W$ に、次の値を入力して得られるインスタンスを解くことを考えます：

| パラメーター | 値 |
| --- | --- |
| $v_{i}$ | `[10, 13, 18, 31, 7, 15]` |
| $w_{i}$ | `[11, 15, 20, 35, 10, 33]` |
| $W$ | `47` |

> **インスタンスとは**: `jijmodeling` では、数理モデルのパラメーターの具体的な値を格納した辞書を"インスタンスデータ"と呼び、数理モデルのパラメーターに具体的な値を入れたものを"インスタンス"と呼んでいます。

## インスタンスの生成手順

`jijmodeling` を使うと、ソルバーに入力するためのインスタンスを次の 3 ステップで生成できます：

1. ナップサック問題を定式化する
2. インスタンスデータを用意する
3. インスタンスを生成する

### Step1. ナップサック問題を定式化する

`jijmodeling` を使用してナップサック問題を定式化すると、以下の Python コードになります：

In [ ]:
import jijmodeling as jm

@jm.Problem.define("Knapsack", sense=jm.ProblemSense.MAXIMIZE)
def knapsack_problem(problem: jm.DecoratedProblem):
    # アイテムの総数
    N = problem.Natural("N")
    # アイテムの価値
    v = problem.Natural("v", shape=N)
    # アイテムの重さ
    w = problem.Natural("w", shape=N)
    # ナップサックの耐荷重
    W = problem.Natural("W")
    # アイテムiをナップサックに入れる場合は1, 入れない場合は0を取る決定変数
    x = problem.BinaryVar("x", shape=(N,))

    # 目的関数
    problem += jm.sum(v[i] * x[i] for i in N)
    # 制約条件: ナップサックの耐荷重を超えない
    problem += problem.Constraint("重量制限", jm.sum(w[i] * x[i] for i in N) <= W)

knapsack_problem

> **ヒント**: `jijmodeling` での定式化の方法については詳しく知りたい場合は [JijModelingの基本](https://jij-inc-jijmodeling-tutorials-ja.readthedocs-hosted.com/ja/latest/basics/overview.html) を参照してください。

### Step2. インスタンスデータを用意する

次に、Step1 で定式化した数理モデルのパラメーター $v_i, w_i, W$ のインスタンスデータを用意します。

In [ ]:
instance_data = {
    "N": 6,
    "v": [10, 13, 18, 31, 7, 15],  # アイテムの価値のデータ
    "w": [11, 15, 20, 35, 10, 33], # アイテムの重さのデータ
    "W": 47,                       # ナップサックの耐荷重のデータ
}

### Step3. インスタンスに変換する

最後に、定式化した数理モデルと用意したインスタンスデータを用いてインスタンスを生成しましょう。

In [ ]:
instance = knapsack_problem.eval(instance_data)

> **ヒント**: `Problem.eval` の返却値は `ommx.v1.Instance` オブジェクトです。詳しくは [インスタンスの生成](https://jij-inc-jijmodeling-tutorials-ja.readthedocs-hosted.com/ja/latest/basics/instance_generation.html) を参照してください。

## 最適化問題を解く

では、Step3 で得られたインスタンスを JijZept Solver で解いてみましょう。

In [ ]:
import jijzept_solver

# JijZept Solverを介して問題を解き、ommx.v1.Solutionとして解を取得
solution = jijzept_solver.solve(instance, time_limit=3.0)

### 結果の確認

目的関数値は `objective` プロパティで確認できます。

In [ ]:
solution.objective

また、得られた解は `decision_variables_df` プロパティを使うことで `pandas.DataFrame` オブジェクトとして確認することができます:

In [ ]:
solution.decision_variables_df[["name", "subscripts", "value"]]